In [1]:
import datasets
import numpy as np
import os
from PIL import Image
from datasets import Image as Image_ds
from datasets import ClassLabel, Features
import pandas as pd 
import torch
from tqdm import tqdm
from collections import Counter
import os
from torch import optim, nn, utils, Tensor
from torchvision.transforms import ToTensor
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from torch.utils.data import DataLoader

/work/art-multimodal-benchmark/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = datasets.load_from_disk(os.path.join('..', 'data', 'wikiart_filtered_remapped_FINAL'))

In [3]:
def remap_features(ds_original, ds_filtered, label):

    original_feature = ds_original.features[label] # the ClassLabel feature
    original_names = original_feature.names

    # classes(names) in new subclassification dataset:
    used_class_names = sorted(list(set(ds_filtered[f"{label}_str"])))
    new_class_label = ClassLabel(names=used_class_names)

    # set up function to remap from str -> int for new ClassLabels
    def remap_labels(example):
        example[label] = new_class_label.str2int(example[f"{label}_str"])
        return example
    
    # use map to remap classlabels
    ds_filtered = ds_filtered.map(remap_labels)

    # recast the class label feature to new labels
    new_features = ds_filtered.features.copy()
    new_features[label] = new_class_label
    ds_filtered = ds_filtered.cast(new_features)

    return ds_filtered

In [4]:
# write function that allows to filter data based on input parameters:

def filter_data(ds, label, subclassification_task, seed):
    ds = ds.add_column('old_indices', range(len(ds)))

    # find the rows that matches the subclassification task
    subclass_indices = [idx for idx, a in enumerate(ds[f'{label}_str']) if a in subclassification_task]
    ds_subset = ds.select(subclass_indices)

    # remap labels to fit to new number of classes for subclassification task
    ds_subset = remap_features(ds, ds_subset, label)

    # split into train, val and test: 
    ds_split = ds_subset.train_test_split(test_size=0.2, seed=seed, stratify_by_column = label)
    ds_train = ds_split['train']
    ds_test = ds_split['test']

    # split test data into test and validation
    ds_test_split = ds_test.train_test_split(test_size=0.5, seed=seed, stratify_by_column = label)
    ds_val = ds_test_split['train']
    ds_test = ds_test_split['test']

    ds_splits = {
            'train': ds_train,
            'test': ds_test,
            'val': ds_val}

    return ds_splits

In [5]:
rembrandt_vs_rubens = ['rembrandt', 'peter-paul-rubens']
ds_splits = filter_data(ds, 'artist', rembrandt_vs_rubens, 2830)

Casting the dataset: 100%|██████████| 891/891 [00:00<00:00, 1758.06 examples/s]


### Test with PyTorch lightning

In [6]:
# figure out how to make these into parameters instead
hidden_layer_size = 200
label = 'artist'
inp_size = 1014 # dims of embedding
dropout_p = 0.3
device = 'cpu'

num_classes = ds_splits['train'].features[label].num_classes

model = nn.Sequential(
        nn.Linear(in_features=inp_size, out_features=hidden_layer_size),
        nn.ReLU(),
        nn.Dropout(p=dropout_p),
        nn.Linear(in_features=hidden_layer_size, out_features=num_classes)
            ).to(device)

In [ ]:
# what exactly is this doing
L.seed_everything(1121218)

In [7]:
def define_class_weights(ds_splits, label):
    y_tensor = torch.tensor(ds_splits['train'][label])
    class_counts = torch.bincount(y_tensor)
    class_weights = 1.0 / class_counts.float() # weight the loss inversely proportional to class frequency
    class_weights /= class_weights.sum() # normalize weights so they sum to one

    return class_weights

In [8]:
# define a LightningModule

# define the model with a build in training loop, validation loop, test loop + optimizer
class SubclassModel(L.LightningModule):
    def __init__(self, model, class_weights, lr, weight_decay): # options to set some default parameters here

        # not really sure what this does:
        super().__init__()

        self.model = model
        self.lr = lr 
        self.weight_decay = weight_decay 

        # buffer makes sure that class weights moves automatically to GPU
        self.register_buffer('class_weights', class_weights)
        self.loss_fn = nn.CrossEntropyLoss(weight=self.class_weights)
    
    # not exactly sure what this part is
    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        X, y = batch 
        output = model(X)
        loss = self.loss_fn(output, y)
        acc = (output.argmax(1) == y).float().mean()

        # log the training loss and accuracy
        # Log the loss at each training step and epoch, create a progress bar
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True, logger=True) # logged per-epoch level
        self.log("train_acc", acc, on_step=True, on_epoch=True, prog_bar=True, logger=True)

        return loss 
    
    # this defines the validation loop
    def validation_step(self, batch, batch_idx):
        X, y = batch 
        output = model(X)
        loss = self.loss_fn(output, y)
        acc = (output.argmax(1) == y).float().mean()
        self.log('val_loss', loss)
        self.log('val_acc', acc)
    
    # lightning automatically runs testing with torch.no_grad() and model.eval()
    def test_step(self, batch, batch_idx):
        X, y = batch 
        output = model(X)
        loss = self.loss_fn(output, y)
        acc = (output.argmax(1) == y).float().mean()
        self.log('test_loss', loss)
        self.log('test_acc', acc) 

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9) # set gamma or make changeble parameter?

        return { # has to be returned in a specific format
                "optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": scheduler,
                    "monitor": "val_loss",
                },
                }


In [13]:
# define callbacks and loggers:

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

file_name = "rembrandt_vs_rubens"
model_name = 'clip-vit-large-patch14'

checkpoint_callback = ModelCheckpoint(
   dirpath=os.path.join('..', 'checkpoints', model_name), # change with actual path for running script
   monitor="val_loss",
   filename=file_name + "-{epoch:02d}-{val_loss:.2f}-{val_acc:.2f}",
   save_top_k=2,
   mode="min",
)

#logger = TensorBoardLogger(save_dir="lightning_logs", name=file_name)

early_stopping = EarlyStopping(monitor="val_loss", patience=5, mode="min", verbose=False)

In [ ]:
def create_dataloader(ds_splits, full_embedding_pt, label, split, batch_size, idx_column):
    
    class EmbeddingsDataset(Dataset):
        def __init__(self, embeddings, labels):
            self.embeddings = embeddings
            self.labels = labels

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return self.embeddings[idx], self.labels[idx]

    # load full embedding and split based on correct indices
    split_indices = ds_splits[split][idx_column]
    #full_embedding_pt = torch.load(os.path.join('data', 'filtered_embeddings_FINAL', model_name, f'{model_name}_all_splits.pt'))

    filtered_embeddings = full_embedding_pt[split_indices]

    # cast to float32
    #embeddings_tensor = filtered_embeddings.float().to(device)
    embeddings_tensor = filtered_embeddings.float()

    y = ds_splits[split][label]
    labels_tensor = torch.tensor(y)

    shuffle=False

    if split == 'train':
        shuffle=True

    dataset = EmbeddingsDataset(embeddings_tensor, labels_tensor)

    # input to data loader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle) # set shuffle=True for train

    embedding_size = embeddings_tensor.shape[1]

    return dataloader, embedding_size

In [ ]:
# now model is set up, try to use dataloaders now:
# define dataloaders

full_embedding_pt = torch.load(os.path.join('data', 'filtered_embeddings_FINAL', model_name, f'{model_name}_all_splits.pt'))
train_loader, inp_size = create_dataloader(ds_splits, full_embedding_pt, label, 'train', batch_size, idx_column)
val_loader, _ = create_dataloader(ds_splits, full_embedding_pt, label, 'val', batch_size, idx_column)
test_loader, _ = create_dataloader(ds_splits, full_embedding_pt, label, 'test', batch_size, idx_column)


In [18]:
train_loader

In [ ]:
# important for reproducibility
seed_everything(2830, workers=True) # not entirely sure what workers=True means

In [ ]:
# load trainer
trainer = Trainer(deterministic=True)